# Cross-Validation

## Overview

Cross-validation estimates out-of-sample model performance by partitioning the data into training and validation folds. It is the standard tool for model selection, hyperparameter tuning, and honest performance reporting.

**CV strategies:**

| Strategy | When to use |
|---|---|
| k-Fold (k=5 or 10) | Default for most tabular data |
| Stratified k-Fold | Classification with class imbalance |
| Repeated k-Fold | When variance of CV estimate matters |
| Leave-One-Out (LOO) | Very small n (<50) |
| Group k-Fold | Grouped/clustered data |
| TimeSeriesSplit | Time-ordered data |

**Key principle:** the validation fold must never influence any step of model training — including preprocessing, feature selection, and hyperparameter search.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (KFold, StratifiedKFold, GroupKFold,
    RepeatedKFold, LeaveOneOut, cross_val_score, cross_validate)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, make_scorer
from scipy import stats

rng = np.random.default_rng(42)
n = 200
elevation = rng.uniform(50, 400, n)
nitrate   = rng.gamma(2, 2, n)
richness  = 25 - 0.03*elevation - 0.7*nitrate + rng.normal(0, 3, n)
restored  = (richness > richness.mean()).astype(int)
catchment = rng.choice(range(10), n)   # 10 catchment groups
X = np.column_stack([elevation, nitrate, richness])
print(f"n={n}, class balance: {restored.mean():.2f}")

---
## k-Fold and Stratified k-Fold

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf",    LogisticRegression(random_state=42, max_iter=500))])

kf   = KFold(n_splits=5, shuffle=True, random_state=42)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, cv in [("KFold", kf), ("StratifiedKFold", skf)]:
    scores = cross_val_score(pipe, X, restored, cv=cv, scoring="roc_auc")
    print(f"{name:20s}: AUC={scores.mean():.3f} +/- {scores.std():.3f}  {scores.round(3)}")

print("\nStratified CV ensures each fold has the same class ratio as the full dataset")
print("Always use StratifiedKFold for classification")

---
## Group k-Fold: Preventing Data Leakage from Grouped Data

In [ ]:
gkf = GroupKFold(n_splits=5)
scores_kf  = cross_val_score(pipe, X, restored, cv=kf,  scoring="roc_auc")
scores_gkf = cross_val_score(pipe, X, restored, cv=gkf,
                              groups=catchment, scoring="roc_auc")
print(f"KFold (ignores groups):    AUC={scores_kf.mean():.3f} +/- {scores_kf.std():.3f}")
print(f"GroupKFold (respects groups): AUC={scores_gkf.mean():.3f} +/- {scores_gkf.std():.3f}")
print()
print("When observations from the same catchment appear in both train and test,")
print("the model can memorise catchment-level patterns -> inflated AUC.")
print("GroupKFold ensures each catchment appears in only one fold.")

---
## CV with Pipeline: No Leakage from Preprocessing

In [ ]:
# WRONG: fit scaler on all data before CV
scaler_leak = StandardScaler().fit(X)
X_scaled_leak = scaler_leak.transform(X)
clf = LogisticRegression(random_state=42, max_iter=500)
scores_leak = cross_val_score(clf, X_scaled_leak, restored,
                               cv=skf, scoring="roc_auc")

# CORRECT: scaler inside pipeline, fit only on training fold
pipe_correct = Pipeline([("scaler", StandardScaler()),
                          ("clf", LogisticRegression(random_state=42, max_iter=500))])
scores_correct = cross_val_score(pipe_correct, X, restored,
                                  cv=skf, scoring="roc_auc")

print(f"Leaky preprocessing:   AUC={scores_leak.mean():.3f} +/- {scores_leak.std():.3f}")
print(f"Pipeline (correct):    AUC={scores_correct.mean():.3f} +/- {scores_correct.std():.3f}")
print()
print("For this clean dataset the difference is small, but with high-d data")
print("and feature selection the leak can be 10-20 AUC points.")

---
## Comparing Models with CV

In [ ]:
models = {
    "Logistic Regression": Pipeline([("sc", StandardScaler()),
        ("clf", LogisticRegression(random_state=42, max_iter=500))]),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
}
cv_results = {}
for name, model in models.items():
    res = cross_validate(model, X, restored, cv=skf,
                         scoring=["roc_auc","average_precision"],
                         return_train_score=True)
    cv_results[name] = res
    print(f"{name}:")
    print(f"  Train AUC: {res['train_roc_auc'].mean():.3f} | Test AUC: {res['test_roc_auc'].mean():.3f} +/- {res['test_roc_auc'].std():.3f}")
    print(f"  Train AP:  {res['train_average_precision'].mean():.3f} | Test AP:  {res['test_average_precision'].mean():.3f}")
print()
print("Large train-test gap -> overfitting. Prefer model with smaller gap at similar test AUC.")

---

## Common Pitfalls

**1. Fitting any preprocessing step on the full dataset before CV**  
Fitting a scaler, PCA, or feature selector on all n observations before cross-validation leaks validation set information into training. Always wrap preprocessing inside a `Pipeline` so it is fitted only on the training fold at each split.

**2. Using standard KFold for grouped or clustered data**  
When observations from the same group (catchment, patient, site) can appear in both training and test folds, the model learns group-level patterns and performance is optimistically inflated. Always use `GroupKFold` when the data has a meaningful grouping structure.

**3. Reporting only CV mean without standard deviation**  
CV mean alone is insufficient — a model with AUC 0.85 ± 0.12 is not reliably better than one with AUC 0.82 ± 0.02. Always report both mean and SD of CV scores, and consider whether the difference between models exceeds 1 SD.

**4. Using LOO-CV for large datasets**  
Leave-one-out is exact but requires fitting n models. For n > 200 it is computationally expensive with negligible benefit over 10-fold CV. Reserve LOO for very small datasets (n < 50).

**5. Tuning hyperparameters with CV and then reporting the same CV score as the test performance**  
If CV is used to select hyperparameters, the CV score is optimistically biased for the selected configuration. Use nested CV (inner loop for tuning, outer loop for evaluation) or a held-out test set for honest performance reporting.
---
*python_methods_library - Samantha McGarrigle*